# CNN in PyTorch — live coding lab

**Course:** Master's Degree in Computer Engineering  
**Format:** live coding + short individual activities  
**Reference:** `PyTorch/12_deep_computer_vision_with_cnns.ipynb` from the DeepLearning repository.

## Goals
By the end of the lab, students will be able to:
1. clearly distinguish **CPU, GPU, and data transfers** in PyTorch;
2. reason about the **shapes** produced by convolutions and pooling;
3. build a compact CNN for Fashion-MNIST;
4. write a correct, device-aware training loop;
5. diagnose a model using accuracy and selected misclassifications.

> **Class method:** before running a cell, predict the result, tensor shape, or behavior. Cells marked **LIVE** are completed together; cells marked **ACTIVITY** are short individual exercises.


In [ ]:
import time
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)

# Colab: Runtime > Change runtime type > T4 GPU (se disponibile)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 1. CPU → GPU: what does it really mean? **[LIVE]**

A tensor lives on a *device*. Operations require data and model parameters to be on the same device.

### Predict before running
What do you expect from these three operations?
- create `A` and `B` on the CPU;
- copy them to the GPU with `.to(device)`;
- multiply them with `A @ B`.

For meaningful CUDA benchmarking, two details matter:
- **warm-up**, because the first operation includes initialization overhead;
- `torch.cuda.synchronize()` before stopping the timer, because CUDA kernels execute asynchronously.


In [ ]:
N = 4096
A_cpu = torch.randn(N, N)
B_cpu = torch.randn(N, N)

# TODO 1 — misura il tempo della moltiplicazione su CPU
# Suggerimento: time.perf_counter()

if device.type == "cuda":
    # TODO 2 — move A_cpu and B_cpu to the GPU and measure transfer time

    # TODO 3 — esegui UNA moltiplicazione di warm-up e sincronizza CUDA

    # TODO 4 — misura A_gpu @ B_gpu ricordando torch.cuda.synchronize()

    # TODO 5 — riporta il risultato su CPU e confrontalo con quello CPU
    pass
else:
    print("Nessuna GPU CUDA: proseguiamo su", device)


### ACTIVITY 1 — transfer or computation?

Predict what happens if `N` becomes much smaller, for example `N=256`.

**Question:** is the GPU still necessarily faster *end-to-end*? Why can transfer cost dominate?

> Key message: a GPU is most useful when we perform **a lot of computation per byte transferred** and, during training, keep both model and mini-batches on the GPU instead of repeatedly moving data CPU↔GPU.


## 2. Dataset and mini-batches: inspect before modeling **[LIVE]**

We will use **Fashion-MNIST**: 28×28 grayscale images, 10 classes. It is small enough for live training but non-trivial enough to show why CNNs are useful.

Before running the cell: what shape do you expect for a mini-batch of 128 images?


In [ ]:
transform = transforms.ToTensor()

train_full = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST("data", train=False, download=True, transform=transform)

# Manteniamo una validazione separata.
g = torch.Generator().manual_seed(SEED)
train_data, valid_data = torch.utils.data.random_split(train_full, [55_000, 5_000], generator=g)

# FAST_MODE keeps the live session practical and responsive.
FAST_MODE = True
if FAST_MODE:
    train_data = Subset(train_data, range(12_000))
    valid_data = Subset(valid_data, range(2_000))
    test_live = Subset(test_data, range(2_000))
else:
    test_live = test_data

BATCH_SIZE = 128
pin = device.type == "cuda"
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)
test_loader = DataLoader(test_live, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)

class_names = train_full.classes
X, y = next(iter(train_loader))
print("X:", X.shape, X.dtype, "range:", (X.min().item(), X.max().item()))
print("y:", y.shape, y.dtype)
print(class_names)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, label in zip(axes, X[:8], y[:8]):
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()


## 3. A convolution before the CNN **[LIVE]**

Before using `nn.Conv2d`, let us make the operation visible. We create a synthetic image with a vertical edge and apply a kernel that measures horizontal intensity differences.

For an input `[B, C, H, W]`, `F.conv2d` expects a kernel with shape `[C_out, C_in, kH, kW]`.


In [ ]:
img = torch.zeros(1, 1, 12, 12)
img[:, :, :, 6:] = 1.0

# TODO 6 — costruisci un kernel 3x3 sensibile a un bordo verticale.
# Partenza suggerita:
# [-1, 0, 1]
# [-1, 0, 1]
# [-1, 0, 1]
# Ricorda poi di ottenere shape [1, 1, 3, 3].

# TODO 7 — applica F.conv2d(..., padding=1)



fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(img[0, 0], cmap="gray")
axes[0].set_title("input")
axes[1].imshow(kernel[0, 0], cmap="gray")
axes[1].set_title("kernel")
axes[2].imshow(response[0, 0], cmap="gray")
axes[2].set_title("feature map")
for ax in axes: ax.axis("off")
plt.tight_layout()
print("input:", img.shape, "kernel:", kernel.shape, "output:", response.shape)

## 4. Shape reasoning: padding, stride, pooling **[ACTIVITY]**

For one spatial dimension:

\[
H_{out}=\left\lfloorrac{H+2P-K}{S}
ight
floor+1
\]

### ACTIVITY 2 — paper and pencil
For an input of size `28×28`, compute the shapes after:
1. `Conv2d(1, 16, kernel_size=3, padding=1)`;
2. `MaxPool2d(2)`;
3. `Conv2d(16, 32, kernel_size=3, padding=1)`;
4. `MaxPool2d(2)`.

How many features enter the first linear layer?


In [ ]:
probe = torch.randn(4, 1, 28, 28)
blocks = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
)

z = probe
for layer in blocks:
    z = layer(z)
    print(f"{layer.__class__.__name__:12s} -> {tuple(z.shape)}")
print("flatten features/image:", z[0].numel())


## 5. Build the CNN **[LIVE]**

Compact architecture:

`[1,28,28] → Conv(16) → Pool → Conv(32) → Pool → Flatten → 64 → 10`

Questions to ask before completing the code:
- why is the first `in_channels` equal to 1?
- what does `out_channels` represent?
- why do we **not** add Softmax at the output?
- how many parameters does a 3×3 convolution use compared with an equivalent fully connected layer?


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO 9 — features:
        # Conv2d(1,16,3,padding=1) -> ReLU -> MaxPool2d(2)
        # Conv2d(16,32,3,padding=1) -> ReLU -> MaxPool2d(2)
        self.features = nn.Sequential(
            # ...
        )

        # TODO 10 — classifier: Flatten -> Linear(? ,64) -> ReLU -> Linear(64,10)
        self.classifier = nn.Sequential(
            # ...
        )

    def forward(self, x):
        # TODO 11 — forward pass
        pass

# TODO 12 — istanzia il modello e spostalo su device
# model = ...

# Check: for 8 images, the output must have shape [8, 10]


## 6. Device-aware training loop **[LIVE]**

With `CrossEntropyLoss`, the model returns **logits**. Each mini-batch follows:

`zero_grad → forward → loss → backward → optimizer.step`.

The important GPU point is to move **each mini-batch** to `device`, while keeping the model there throughout training.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X, y in loader:
        # TODO 13 — sposta X e y su device (non_blocking=True se vuoi sfruttare pin_memory)

        if training:
            # TODO 14 — azzera i gradienti
            pass

        with torch.set_grad_enabled(training):
            # TODO 15 — forward + loss

            if training:
                # TODO 16 — backward + optimizer step
                pass

        # TODO 17 — accumula loss, correttezza e numero esempi

    # TODO 18 — restituisci loss media e accuracy
    raise NotImplementedError

# TODO 19 — loss e optimizer
# criterion = ...
# optimizer = ...


In [ ]:
EPOCHS = 3 if FAST_MODE else 5
history = []

# TODO 20 — per ogni epoca:
# 1) run_epoch(..., optimizer) sul train
# 2) run_epoch(...) sulla validation
# 3) stampa loss/accuracy e tempo dell'epoca


## 7. Evaluation: more than a single accuracy value **[LIVE]**

Inspect a few misclassified examples. They are often more informative than a single metric.


In [ ]:
# TODO 21 — valuta il modello sul test set

# TODO 22 — find some misclassified images and visualize:
# image, true class, predicted class


## 8. ACTIVITY 3 — ablation challenge **[ACTIVITY]**

Work individually or in pairs. Choose **one** modification, state a hypothesis first, then test it:

- **A.** replace the second `MaxPool2d(2)` with `Identity` and fix the input size of the `Linear` layer;
- **B.** add `BatchNorm2d` after the convolutional layers;
- **C.** add `Dropout(0.3)` to the classifier;
- **D.** double the channels from `16→32` and `32→64`.

Record three things: **number of parameters**, **training cost**, and **validation accuracy**.
The goal is not to find a single “winner”, but to explain **why** the change affects capacity, computational cost, and generalization.


## 9. Wrap-up

Three ideas to take away:
1. **GPU ≠ magic:** placement, transfers, and synchronization matter;
2. in CNNs, reasoning about **tensor shapes** prevents many errors and clarifies the role of kernels, padding, and pooling;
3. a CNN reuses weights across space: **locality + weight sharing** are the structural reasons it is well suited to images.

### Exit ticket
Write one sentence for each question:
- when can a CPU→GPU transfer cancel out the benefit of GPU computation?
- why does `CrossEntropyLoss` expect logits instead of probabilities already passed through Softmax?
- what is the feature shape immediately before `Flatten` in our network?
